# Agentic AI

## 실습 전 Tavily API 셋팅
- https://tavily.com 에서 발급 (무료 크레딧 제공)

`.env`

```
TAVILY_API_KEY=tvly-...
````

## Step 01 · 환경 설정

In [ ]:
%pip install -U langchain-tavily ddgs requests langchain-community

In [ ]:
import json
import os

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

print("OpenAI:", "OK" if os.getenv("OPENAI_API_KEY") else "없음", "| Tavily:", "OK" if os.getenv("TAVILY_API_KEY") else "없음")

MODEL = "gpt-6-luna"
llm = ChatOpenAI(model=MODEL)

## Step 02 · 사용자 도구 정의

### 2-1. @tool
- 타입 힌트 = 입력 스키마
- docstring = 도구 설명

In [ ]:
from langchain.tools import tool

from datetime import date, timedelta

@tool
def add_business_days(start_date: str, days: int) -> str:
    """시작 날짜(YYYY-MM-DD)에서 주말을 제외한 영업일 기준으로 days일 뒤의 날짜를 계산합니다."""

    d = date.fromisoformat(start_date)
    added = 0
    while added < days:
        d += timedelta(days=1)
        if d.weekday() < 5:
            added += 1
    return d.isoformat()

In [ ]:
print("name       :", add_business_days.name)
print("description:", add_business_days.description)
print("args       :", add_business_days.args)
print("invoke     :", add_business_days.invoke({"start_date": "2026-10-02", "days": 3}))

### 2-2. 이름 · 설명 직접 지정

In [ ]:
@tool("unit_converter", description="길이 단위를 변환합니다. km, m, mile 사이의 변환에 사용하세요.")
def convert(value: float, from_unit: str, to_unit: str) -> str:
    """(description 인자가 docstring보다 우선)"""
    to_m = {"km": 1000, "m": 1, "mile": 1609.344}
    return f"{value * to_m[from_unit] / to_m[to_unit]:.3f} {to_unit}"

In [ ]:
print(convert.name, "|", convert.description)

In [ ]:
print(convert.invoke({"value": 10, "from_unit": "km", "to_unit": "mile"}))

### 2-3. args_schema 

- Pydantic으로 입력 형식 · 허용값 정의

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field

class ExchangeInput(BaseModel):
    amount: float = Field(description="변환할 금액")
    currency: Literal["USD", "JPY", "EUR"] = Field(description="변환할 외화 통화 코드")

RATES = {"USD": 1400.0, "JPY": 9.3, "EUR": 1510.0}   # 예시 환율 (원)

In [ ]:
@tool(args_schema=ExchangeInput)
def to_krw(amount: float, currency: str) -> dict:
    """외화 금액을 원화로 환산합니다. 결과는 dict로 반환합니다."""
    return {
        "amount": amount, 
        "currency": currency, 
        "krw": round(amount * RATES[currency]), 
        "rate": RATES[currency]
    }

In [ ]:
print(json.dumps(to_krw.args, ensure_ascii=False, indent=1))

In [ ]:
print(to_krw.invoke({"amount": 50, "currency": "USD"}))

## Step 03 · Tool Calling 루프 

(에이전트 없이 직접 적용하는 방법)

1. `bind_tools`로 모델에 도구 목록 전달 → 
2. 모델이 `tool_calls` 생성 → 
3. 코드가 도구 실행 → 
4. `ToolMessage`로 결과 전달 → 
5. 모델이 최종 답변


In [ ]:
from langchain.messages import HumanMessage, ToolMessage

tools = [add_business_days, convert, to_krw]
tools_by_name = {t.name: t for t in tools}
llm_with_tools = llm.bind_tools(tools)

In [ ]:
messages = [
    HumanMessage("2026-10-02부터 영업일 5일 뒤는 며칠이야? 그리고 30달러는 원화로 얼마야?")
]
ai = llm_with_tools.invoke(messages)
print("tool_calls:", ai.tool_calls)

In [ ]:
messages.append(ai)
for call in ai.tool_calls:
    result = tools_by_name[call["name"]].invoke(call["args"])
    print(f"  ↳ {call['name']}({call['args']}) → {result}")
    messages.append(ToolMessage(content=str(result), tool_call_id=call["id"]))

In [ ]:
final = llm_with_tools.invoke(messages)
print("\n최종 답변:", final.content)

## Step 04 · 웹 검색 도구

### 4-1. TavilySearch 

- LangChain 공식 통합 (AI용 검색 API)

In [ ]:
from langchain_tavily import TavilySearch

tavily = TavilySearch(max_results=3, topic="general")

In [ ]:
res = tavily.invoke({"query": "LangChain 1.0 release"})
for r in res.get("results", []):
    print("-", r["title"], "|", r["url"])

### 4-2. DDGS 
- API Key 없이 쓰는 검색을 커스텀 도구로 감싸기

In [ ]:
from ddgs import DDGS

@tool
def web_search(query: str, max_results: int = 3) -> str:
    """웹에서 최신 정보를 검색합니다. 뉴스 · 최신 동향 · 모델이 모르는 사실을 확인할 때 사용하세요.
    결과는 제목, URL, 요약으로 반환됩니다."""
    try:
        hits = DDGS().text(query, max_results=max_results)
    except Exception as e:
        return f"검색 실패: {e}"
    if not hits:
        return "검색 결과가 없습니다."
    return "\n\n".join(f"[{h['title']}]({h['href']})\n{h['body']}" for h in hits)

print(web_search.invoke({"query": "OpenAI GPT-6 출시"})[:500])

### 4-3. Serper API (Google)

- [Serper](https://serper.dev/)에서 회원가입
     - 초기 2500 크레딧 무료

In [ ]:
from langchain_community.utilities import GoogleSerperAPIWrapper

# type 파라미터를 통해 news, images, places를 검색가능
search = GoogleSerperAPIWrapper(serper_api_key='9e8d0b4558b77fe8303b56ee6b2dc9fcf6917393')

In [ ]:
query = "2026년 파이썬 최신 프레임워크 트렌드"
result = search.run(query)

In [ ]:
print(f"검색 결과:\n{result}")

In [ ]:
# 구글 이미지 검색

import pprint
search = GoogleSerperAPIWrapper(type="images", serper_api_key='9e8d0b4558b77fe8303b56ee6b2dc9fcf6917393')
results = search.results("Lion")
pprint.pp(results)

In [ ]:
# 구글 뉴스 검색

search = GoogleSerperAPIWrapper(type="news", serper_api_key='9e8d0b4558b77fe8303b56ee6b2dc9fcf6917393')
results = search.results("Tesla Inc.")
pprint.pp(results)

In [ ]:
# tbs 파라미터를 활용해 검색 결과의 시간을 제한 가능
# qdr:h (past hour) qdr:d (past day) qdr:w (past week) qdr:m (past month) qdr:y (past year)

# 특정 기간을 설정하는것도 가능
# qdr:h12 (past 12 hours)
# qdr:d3, qdr:w2, qdr:m6, qdr:y2
search = GoogleSerperAPIWrapper(type="news", tbs="qdr:h", serper_api_key='9e8d0b4558b77fe8303b56ee6b2dc9fcf6917393')
results = search.results("Tesla Inc.")
pprint.pp(results)

In [ ]:
# 구글 장소 검색

search = GoogleSerperAPIWrapper(type="places", serper_api_key='9e8d0b4558b77fe8303b56ee6b2dc9fcf6917393')
results = search.results("Italian restaurants in Upper East Side")
pprint.pp(results)

### 4-4. 검색도구 Chaining

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [ ]:
template = """You are an AI assistant. Answer the question.
If you don't know the answer, just say you don't know.

Question: {question}
Answer:"""

prompt = ChatPromptTemplate.from_template(template)
parser = StrOutputParser()

chain = prompt | llm | parser

In [ ]:
# gpt 6 luna의 지식 단절은 May 18, 2026
result = chain.invoke({"question": "When is the release date for the GPT-6.1 Sol model?"})
print(result)

In [ ]:
template = """Answer the question based on context.

Question: {question}
Context: {context}
Answer:"""

prompt = ChatPromptTemplate.from_template(template)
parser = StrOutputParser()

chain = (
    {"question": RunnablePassthrough(), "context": RunnablePassthrough()}
    | prompt 
    | llm 
    | parser
)

In [ ]:
# 검색 결과 확인
search_result = tavily.run("When is the release date for the GPT-6.1 Sol model?")

print(search_result)

In [ ]:
question = "When is the release date for the GPT-6.1 Sol model?"

result = chain.invoke({"question": question, "context": tavily.run(question)})
print(result)

## Step 05 · 데이터베이스 쿼리 도구

예제용 SQLite 인메모리 DB 
- 조회(SELECT)만 허용하는 가드 포함

In [ ]:
import sqlite3

# 예제용 데이터베이스 셋팅
conn = sqlite3.connect(":memory:", check_same_thread=False)

conn.executescript("""
CREATE TABLE products (id INTEGER PRIMARY KEY, name TEXT, category TEXT, price INTEGER, stock INTEGER);
CREATE TABLE orders (id INTEGER PRIMARY KEY, product_id INTEGER, qty INTEGER, order_date TEXT);
INSERT INTO products VALUES
 (1,'무선 이어폰','전자기기',89000,12),
 (2,'스마트워치','전자기기',199000,0),
 (3,'텀블러','생활용품',29000,40),
 (4,'러닝화','스포츠',119000,7),
 (5,'요가매트','스포츠',35000,25);
INSERT INTO orders VALUES
 (1,1,2,'2026-09-01'),
 (2,3,5,'2026-09-03'),
 (3,4,1,'2026-09-10'),
 (4,1,1,'2026-09-15'),
 (5,5,3,'2026-09-20');
""")

In [ ]:
@tool
def list_tables() -> str:
    """데이터베이스의 테이블 목록과 각 테이블의 컬럼을 반환합니다. SQL을 작성하기 전에 먼저 호출하세요."""
    rows = conn.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()
    out = []
    for (name,) in rows:
        cols = [c[1] for c in conn.execute(f"PRAGMA table_info({name})")]
        out.append(f"{name}({', '.join(cols)})")
    return "\n".join(out)

In [ ]:
@tool
def run_sql(query: str) -> str:
    """SQLite SELECT 쿼리를 실행하고 결과를 반환합니다. 조회(SELECT)만 가능하며 최대 20행을 반환합니다."""
    q = query.strip().rstrip(";")
    if not q.lower().startswith("select") or ";" in q:
        return "오류: 단일 SELECT 쿼리만 실행할 수 있습니다."
    try:
        cur = conn.execute(q)
        cols = [d[0] for d in cur.description]
        rows = cur.fetchmany(20)
    except sqlite3.Error as e:
        return f"SQL 오류: {e}. 테이블 · 컬럼 이름을 list_tables로 확인하세요."
    return json.dumps([dict(zip(cols, r)) for r in rows], ensure_ascii=False)

In [ ]:
print(list_tables.invoke({}))

In [ ]:
print(run_sql.invoke({"query": "SELECT name, stock FROM products WHERE stock = 0"}))
print(run_sql.invoke({"query": "DELETE FROM products"})) # 가드 확인

## Step 06 · 날씨 조회 API 도구 

- Open-Meteo (API Key 불필요)

In [ ]:
import requests

WEATHER_CODE = {0: "맑음", 1: "대체로 맑음", 2: "구름 조금", 3: "흐림", 45: "안개", 51: "이슬비",
                61: "비", 63: "비", 65: "강한 비", 71: "눈", 80: "소나기", 95: "뇌우"}

In [ ]:
class WeatherInput(BaseModel):
    city: str = Field(description="도시 이름 (영문 권장, 예: Seoul, Busan, Tokyo)")

In [ ]:
@tool(args_schema=WeatherInput)
def get_weather(city: str) -> dict:
    """도시의 현재 기온 · 날씨 · 풍속을 조회합니다."""
    geo = requests.get("https://geocoding-api.open-meteo.com/v1/search",
                       params={"name": city, "count": 1, "language": "ko"}, timeout=10).json()
    if not geo.get("results"):
        return {"error": f"'{city}' 위치를 찾을 수 없습니다. 영문 도시 이름으로 다시 시도하세요."}
    loc = geo["results"][0]
    cur = requests.get("https://api.open-meteo.com/v1/forecast", params={
        "latitude": loc["latitude"], "longitude": loc["longitude"],
        "current": "temperature_2m,weather_code,wind_speed_10m",
    }, timeout=10).json()["current"]
    return {
        "city": loc["name"], "country": loc.get("country"),
        "temperature_c": cur["temperature_2m"],
        "weather": WEATHER_CODE.get(cur["weather_code"], f"code {cur['weather_code']}"),
        "wind_kmh": cur["wind_speed_10m"],
    }

In [ ]:
print(get_weather.invoke({"city": "Seoul"}))

## Step 07 · 에이전트 - create_agent

### 7-1. 도구를 모두 연결한 에이전트

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[web_search, list_tables, run_sql, get_weather, to_krw],
    system_prompt=(
        "당신은 쇼핑몰 운영을 돕는 어시스턴트입니다. "
        "재고 · 주문은 데이터베이스 도구로, 날씨는 날씨 도구로, 최신 정보는 웹 검색으로 확인하세요. "
        "도구 결과에 근거해 한국어로 간결하게 답하세요."
    ),
)

In [ ]:
result = agent.invoke({
    "messages": [
        {"role": "user", "content": "재고가 10개 미만인 상품과 가격을 알려줘"}
    ]})

print(result["messages"][-1].content)

In [ ]:
# 에이전트가 거친 단계 확인
for m in result["messages"]:
    kind = type(m).__name__
    if getattr(m, "tool_calls", None):
        print(f"{kind}: tool_calls → {[(c['name'], c['args']) for c in m.tool_calls]}")
    else:
        print(f"{kind}: {str(m.content)[:100]}")

### 7-2. 스트리밍 

- 단계별 진행 상황 보기

In [ ]:
question = {
    "messages": [
        {"role": "user", "content": "서울 날씨를 보고, 스포츠 카테고리 상품 중 오늘 추천할 만한 것을 골라줘"}
    ]}

for chunk in agent.stream(question, stream_mode="updates"):
    for node, update in chunk.items():
        last = update["messages"][-1]
        if getattr(last, "tool_calls", None):
            print(f"[{node}] 도구 호출:", [c["name"] for c in last.tool_calls])
        else:
            print(f"[{node}]", str(last.content)[:150])

### 7-3. 메모리 

- checkpointer + thread_id

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

memory_agent = create_agent(
    model=llm, tools=[get_weather],
    system_prompt="당신은 여행 도우미입니다.",
    checkpointer=InMemorySaver(),
)

In [ ]:
cfg = {"configurable": {"thread_id": "trip-001"}}
memory_agent.invoke({"messages": [{"role": "user", "content": "다음 주에 부산 여행 가요."}]}, cfg)

In [ ]:
r = memory_agent.invoke({"messages": [{"role": "user", "content": "거기 지금 날씨 어때?"}]}, cfg)
print(r["messages"][-1].content)

In [ ]:
other = memory_agent.invoke({"messages": [{"role": "user", "content": "거기 지금 날씨 어때?"}]},
                            {"configurable": {"thread_id": "trip-002"}})
print("\n[다른 thread]", other["messages"][-1].content)

## Step 08 · 고급

### 8-1. ToolRuntime 
- 모델에게 보이지 않는 실행 정보(context) 사용


In [ ]:
from dataclasses import dataclass
from langchain.tools import ToolRuntime

@dataclass
class UserContext:
    user_id: str
    grade: Literal["basic", "vip"]

In [ ]:
@tool
def my_discount(price: int, runtime: ToolRuntime[UserContext]) -> str:
    """현재 사용자의 회원 등급 할인을 적용한 가격을 계산합니다."""
    rate = 0.1 if runtime.context.grade == "vip" else 0.0
    return f"{runtime.context.user_id}({runtime.context.grade}) 할인가: {int(price * (1 - rate)):,}원"

print("모델에 보이는 인자:", my_discount.args)   # runtime은 숨겨짐

In [ ]:
ctx_agent = create_agent(
    model=llm, 
    tools=[my_discount, run_sql], 
    context_schema=UserContext,
    system_prompt="가격은 DB에서 조회하고, 할인은 my_discount로 계산하세요."
)

In [ ]:
r = ctx_agent.invoke({
    "messages": [{"role": "user", "content": "러닝화 내 할인가 알려줘"}]},
    context=UserContext(user_id="u123", grade="vip")
)
print(r["messages"][-1].content)

### 8-2. 구조화 출력 — response_format

In [ ]:
class WeatherBrief(BaseModel):
    city: str
    temperature_c: float
    outfit: str = Field(description="기온에 맞는 옷차림 추천 한 문장")

In [ ]:
brief_agent = create_agent(model=llm, tools=[get_weather], response_format=WeatherBrief)

In [ ]:
r = brief_agent.invoke({"messages": [{"role": "user", "content": "도쿄 날씨 기준으로 옷차림 추천해줘"}]})
print(r["structured_response"])

### 8-3. 미들웨어

- 도구 재시도 · 호출 횟수 제한

In [ ]:
from langchain.agents.middleware import ToolCallLimitMiddleware, ToolRetryMiddleware

safe_agent = create_agent(
    model=llm,
    tools=[web_search, get_weather],
    middleware=[
        ToolRetryMiddleware(max_retries=2),                  # 도구 실패 시 재시도
        ToolCallLimitMiddleware(tool_name="web_search", run_limit=3),  # 검색은 1회 실행당 최대 3번
    ],
)

In [ ]:
r = safe_agent.invoke({"messages": [{"role": "user", "content": "LangChain과 LangGraph의 차이를 최신 자료로 정리해줘"}]})
print(r["messages"][-1].content[:500])